In [0]:
# Configuration and Imports

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, FloatType
from pyspark.sql.window import Window
from datetime import datetime

DATABASE_NAME  = "education_db"
BRONZE_TABLE   = f"{DATABASE_NAME}.bronze_enrollment"
SILVER_TABLE   = f"{DATABASE_NAME}.silver_enrollment"
RUN_ID         = datetime.now().strftime("%Y%m%d_%H%M%S")

# Data quality thresholds
MIN_SCORE      = 0.0
MAX_SCORE      = 100.0
MIN_ENROLLMENT = 1
MAX_ENROLLMENT = 500
VALID_YEAR_MIN = 2015
VALID_YEAR_MAX = 2025
MAX_DROPOUT    = 1.0
HIGH_DROPOUT   = 0.08
MED_DROPOUT    = 0.05

spark.sql(f"USE {DATABASE_NAME}")
print(f"Silver target: {SILVER_TABLE}")
print(f"Run ID: {RUN_ID}")

In [0]:
# Load from Bronze Delta Table

df = spark.table(BRONZE_TABLE)
bronze_count = df.count()
print(f"Loaded from Bronze: {bronze_count:,} rows")
df.printSchema()

In [0]:
# Fix #10: Remove Empty Rows

# Remove rows where ALL key business columns are null — these are unfixable
key_cols = ["school_name", "region", "academic_year", "grade", "subject"]
before = df.count()
df = df.dropna(how="all", subset=key_cols)
after = df.count()
print(f"FIX 10a — Empty rows removed: {before - after:,}")
print(f"Remaining rows: {after:,}")


In [0]:
# ── STEP 2: CAST TYPES SAFELY USING try_cast ──────────────────────────────
# try_cast returns NULL for ANY malformed value instead of throwing an error
# This handles: empty strings, 'N/A', 'FY2023', special chars, spaces, etc.

# First create a temp view so we can use SQL try_cast
df.createOrReplaceTempView("bronze_temp")

df = spark.sql("""
    SELECT
        record_id,
        school_name,
        region,
        school_type,

        try_cast(nullif(regexp_extract(academic_year, '(20[0-9]{2})', 1), '') AS INT) AS academic_year,

        CAST(try_cast(nullif(male_count,   '') AS FLOAT) AS INT) AS male_count,
        CAST(try_cast(nullif(female_count, '') AS FLOAT) AS INT) AS female_count,
        CAST(try_cast(nullif(total_enrollment, '') AS FLOAT) AS INT) AS total_enrollment,

        try_cast(nullif(dropout_rate,           '') AS DOUBLE) AS dropout_rate,
        try_cast(nullif(avg_performance_score,  '') AS DOUBLE) AS avg_performance_score,
        try_cast(nullif(pass_rate,              '') AS DOUBLE) AS pass_rate,
        try_cast(nullif(teacher_student_ratio,  '') AS DOUBLE) AS teacher_student_ratio,
        try_cast(nullif(budget_per_student_usd, '') AS DOUBLE) AS budget_per_student_usd,

        grade,
        subject,
        data_quality_flag,
        _ingestion_timestamp,
        _source_file,
        _pipeline_run_id,
        _layer

    FROM bronze_temp
""")

print(f"Cast complete — {df.count():,} rows")

print("Type casting complete using try_cast")
print("Malformed values (empty strings, N/A, etc.) safely converted to null")
print(f"Rows after casting: {df.count():,}")

# Verify null counts after casting
from pyspark.sql import functions as F
df.select(
    F.sum(F.col("male_count").isNull().cast("int")).alias("null_male"),
    F.sum(F.col("female_count").isNull().cast("int")).alias("null_female"),
    F.sum(F.col("avg_performance_score").isNull().cast("int")).alias("null_score"),
    F.sum(F.col("dropout_rate").isNull().cast("int")).alias("null_dropout"),
    F.sum(F.col("academic_year").isNull().cast("int")).alias("null_year")
).display()

In [0]:
# Standardise Text Fields

# School names: strip whitespace, remove leading special characters, title case
df = df.withColumn("school_name",
    F.initcap(F.trim(
        F.regexp_replace(F.col("school_name"), r"^[^a-zA-Z]+|[^a-zA-Z0-9 ]+$", "")
    ))
)

# Regions: strip and title case
df = df.withColumn("region",
    F.initcap(F.trim(F.col("region")))
)

# Grades: normalise 'G1', 'Gr1', 'GRADE 1' all to 'Grade 1'
df = df.withColumn("grade",
    F.when(
        F.upper(F.trim(F.col("grade"))).rlike(r"^G\d+$"),
        F.regexp_replace(F.upper(F.trim(F.col("grade"))), r"G(\d+)", "Grade $1")
    ).when(
        F.upper(F.trim(F.col("grade"))).rlike(r"^GR\.?\d+$"),
        F.regexp_replace(F.upper(F.trim(F.col("grade"))), r"GR\.?(\d+)", "Grade $1")
    ).otherwise(
        F.initcap(F.trim(F.col("grade")))
    )
)

# Subjects: map all variants to 5 canonical subject names
df = df.withColumn("subject",
    F.when(F.upper(F.trim(F.col("subject"))).isin(
        "MATH", "MATHS", "MATHEMATICS", "MATHS."), F.lit("Mathematics"))
    .when(F.upper(F.trim(F.col("subject"))).isin(
        "SCI", "SCIENCE"), F.lit("Science"))
    .when(F.upper(F.trim(F.col("subject"))).isin(
        "ENG", "ENGLISH", "ENGLISH LANGUAGE"), F.lit("English"))
    .when(F.upper(F.trim(F.col("subject"))).isin(
        "SS", "SOCIALS", "SOC.STUDIES", "SOCIAL STUDIES"), F.lit("Social Studies"))
    .when(F.upper(F.trim(F.col("subject"))).isin(
        "ART", "ARTS", "FINE ARTS"), F.lit("Arts"))
    .otherwise(F.initcap(F.trim(F.col("subject"))))
)

print("FIX 5 & 6 — Text standardisation complete")
print("\nDistinct school names after cleaning:")
df.select("school_name").distinct().orderBy("school_name").show(30, truncate=False)


In [0]:
# Fill Missing Numeric Values

# Strategy: fill nulls with the MEDIAN value of the same school group
# Median is preferred over mean — it's not skewed by outliers
# Window function computes the median within each school partition
school_window = Window.partitionBy("school_name")

df = df \
    .withColumn("avg_performance_score",
        F.when(F.col("avg_performance_score").isNull(),
               F.percentile_approx("avg_performance_score", 0.5)
                .over(school_window))
        .otherwise(F.col("avg_performance_score"))
    ) \
    .withColumn("dropout_rate",
        F.when(F.col("dropout_rate").isNull(),
               F.percentile_approx("dropout_rate", 0.5).over(school_window))
        .otherwise(F.col("dropout_rate"))
    ) \
    .withColumn("budget_per_student_usd",
        F.when(F.col("budget_per_student_usd").isNull(),
               F.percentile_approx("budget_per_student_usd", 0.5)
                .over(school_window))
        .otherwise(F.col("budget_per_student_usd"))
    ) \
    .withColumn("pass_rate",
        F.when(F.col("pass_rate").isNull(),
               F.percentile_approx("pass_rate", 0.5).over(school_window))
        .otherwise(F.col("pass_rate"))
    )

print("FIX 1–4 — Missing value imputation complete")
print("\nNull counts after filling (should all be 0 or very low):")
df.select(
    F.sum(F.col("avg_performance_score").isNull().cast("int")).alias("score_nulls"),
    F.sum(F.col("dropout_rate").isNull().cast("int")).alias("dropout_nulls"),
    F.sum(F.col("budget_per_student_usd").isNull().cast("int")).alias("budget_nulls"),
    F.sum(F.col("pass_rate").isNull().cast("int")).alias("pass_rate_nulls")
).display()

In [0]:
# Recalculate Total Enrollment

# Recalculate total_enrollment from male + female
# This fixes the inconsistent_total issue (210 affected rows)
df = df.withColumn("total_enrollment",
    F.when(
        F.col("male_count").isNotNull() & F.col("female_count").isNotNull(),
        F.col("male_count") + F.col("female_count")
    ).otherwise(F.col("total_enrollment"))
)

print("FIX 12 — Total enrollment recalculated from male + female")

# Verify: check for remaining mismatches
mismatches = df.filter(
    F.abs(F.col("total_enrollment") - (F.col("male_count") + F.col("female_count"))) > 1
).count()
print(f"Remaining enrollment mismatches: {mismatches}")


In [0]:
# Remove Outlier Rows

# COMMAND ----------

# ── STEP 6: REMOVE INVALID ROWS ───────────────────────────────────────────
before_filter = df.count()

df = df.filter(
    F.col("total_enrollment").isNotNull() &
    F.col("total_enrollment").between(MIN_ENROLLMENT, MAX_ENROLLMENT) &
    F.col("dropout_rate").isNotNull() &
    F.col("dropout_rate").between(0.0, MAX_DROPOUT) &
    F.col("avg_performance_score").isNotNull() &
    F.col("avg_performance_score").between(MIN_SCORE, MAX_SCORE) &
    F.col("academic_year").isNotNull() &
    F.col("academic_year").between(VALID_YEAR_MIN, VALID_YEAR_MAX) &
    F.col("male_count").isNotNull() &
    F.col("female_count").isNotNull() &
    (F.col("male_count") >= 0) &
    (F.col("female_count") >= 0)
)

after_filter = df.count()
print(f"Rows before filter : {before_filter:,}")
print(f"Rows after filter  : {after_filter:,}")
print(f"Rows removed       : {before_filter - after_filter:,}")


In [0]:
# Remove Duplicates

before_dedup = df.count()

# Deduplicate on business key: school + year + grade + subject
# Keeps the first occurrence of each unique combination
df = df.dropDuplicates(["school_name", "academic_year", "grade", "subject"])

after_dedup = df.count()
print(f"FIX 10 — Deduplication complete")
print(f"  Rows before dedup : {before_dedup:,}")
print(f"  Rows after dedup  : {after_dedup:,}")
print(f"  Duplicates removed: {before_dedup - after_dedup:,}")


In [0]:
# Add Derived Columns

# These new business columns are computed from the clean data
# They add analytical value and are used directly in Gold aggregations

df_silver = df \
    .withColumn("gender_ratio_female",
        F.round(F.col("female_count") / F.col("total_enrollment"), 4)
    ) \
    .withColumn("performance_band",
        F.when(F.col("avg_performance_score") >= 90, "Excellent")
         .when(F.col("avg_performance_score") >= 75, "Good")
         .when(F.col("avg_performance_score") >= 60, "Satisfactory")
         .otherwise("Needs Improvement")
    ) \
    .withColumn("dropout_risk_flag",
        F.when(F.col("dropout_rate") > HIGH_DROPOUT, "HIGH")
         .when(F.col("dropout_rate") > MED_DROPOUT,  "MEDIUM")
         .otherwise("LOW")
    ) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .withColumn("_layer",            F.lit("silver")) \
    .select(
        F.col("record_id").cast(IntegerType()),
        "school_name", "region", "school_type",
        "academic_year", "grade", "subject",
        "male_count", "female_count", "total_enrollment",
        F.round("dropout_rate", 4).alias("dropout_rate"),
        F.round("avg_performance_score", 1).alias("avg_performance_score"),
        F.round("pass_rate", 4).alias("pass_rate"),
        F.round("teacher_student_ratio", 4).alias("teacher_student_ratio"),
        F.round("budget_per_student_usd", 2).alias("budget_per_student_usd"),
        "gender_ratio_female", "performance_band", "dropout_risk_flag",
        "_silver_timestamp", "_layer"
    )

print("Derived columns added:")
print("  gender_ratio_female — female count / total enrollment")
print("  performance_band    — Excellent / Good / Satisfactory / Needs Improvement")
print("  dropout_risk_flag   — HIGH / MEDIUM / LOW")


In [0]:
# MAGIC ## Cell 11 — Write Silver Delta Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

silver_count = df_silver.count()
print("SILVER TRANSFORMATION COMPLETE")
print("=" * 55)
print(f"Table         : {SILVER_TABLE}")
print(f"Bronze rows in: {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rows removed  : {bronze_count - silver_count:,}  ({(bronze_count - silver_count)/bronze_count*100:.1f}%)")


In [0]:
# Verify Silver Data

df_sv = spark.table(SILVER_TABLE)

print(f"Silver row count: {df_sv.count():,}")
print(f"\nFinal Silver schema:")
df_sv.printSchema()

print("\nSample rows (first 5):")
df_sv.select(
    "school_name", "region", "academic_year", "grade", "subject",
    "total_enrollment", "avg_performance_score",
    "dropout_rate", "performance_band", "dropout_risk_flag"
).show(5, truncate=False)

print("\nPerformance band distribution:")
df_sv.groupBy("performance_band").count().orderBy("performance_band").show()

print("\nDropout risk flag distribution:")
df_sv.groupBy("dropout_risk_flag").count().orderBy("dropout_risk_flag").show()


In [0]:
# Data Quality Validation Tests

print("Running 10 data quality tests on Silver table...\n")

tests = {
    "No null school names":       f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE school_name IS NULL OR TRIM(school_name)=''",
    "No null regions":            f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE region IS NULL OR TRIM(region)=''",
    "Valid academic years":       f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE academic_year < {VALID_YEAR_MIN} OR academic_year > {VALID_YEAR_MAX}",
    "Score in 0-100 range":       f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE avg_performance_score < 0 OR avg_performance_score > 100 OR avg_performance_score IS NULL",
    "Dropout rate in 0-1 range":  f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE dropout_rate < 0 OR dropout_rate > 1 OR dropout_rate IS NULL",
    "Positive enrollment":        f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE total_enrollment <= 0 OR male_count < 0 OR female_count < 0",
    "Enrollment math correct":    f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE ABS(total_enrollment - (male_count + female_count)) > 1",
    "No duplicate business keys": f"SELECT COUNT(*) v FROM (SELECT school_name,academic_year,grade,subject,COUNT(*) c FROM {SILVER_TABLE} GROUP BY 1,2,3,4 HAVING c > 1)",
    "Valid risk flags":           f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE dropout_risk_flag NOT IN ('HIGH','MEDIUM','LOW')",
    "Gender ratio 0-1":           f"SELECT COUNT(*) v FROM {SILVER_TABLE} WHERE gender_ratio_female < 0 OR gender_ratio_female > 1",
}

passed, failed = 0, 0
for name, query in tests.items():
    violations = spark.sql(query).collect()[0]["v"]
    icon = "✅" if violations == 0 else "❌"
    status = "PASS" if violations == 0 else f"FAIL ({violations} violations)"
    print(f"  {icon}  {name:<35} {status}")
    if violations == 0: passed += 1
    else: failed += 1

print(f"\n{'='*55}")
print(f"Results: {passed}/{len(tests)} tests passed")
if failed == 0:
    print("Silver data quality verified ✅")